In [2]:
# This is:

# A mini speech recognition model

# Uses RNN (GRU) to understand sequences

# Classifies audio into words


# Step 1: Import libraries
import torch
import torch.nn as nn
import torch.optim as optim

# Step 2: Dummy audio features
X = torch.randn(5, 10, 13)
y = torch.tensor([0, 1, 2, 1, 0])

# Step 3: RNN Model (GRU)
class SpeechRNN(nn.Module):
    def __init__(self):
        super(SpeechRNN, self).__init__()
        self.rnn = nn.GRU(input_size=13, hidden_size=16, batch_first=True)
        self.fc = nn.Linear(16, 3)

    def forward(self, x):
        out, _ = self.rnn(x)
        out = out[:, -1, :]  # last time step
        out = self.fc(out)
        return out

model = SpeechRNN()

# Step 4: Loss + Optimizer
loss_fn = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.01)

# Step 5: Training
model.train()
for epoch in range(50):
    outputs = model(X)
    loss = loss_fn(outputs, y)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

print("Training Done")

# Step 6: Testing
model.eval()
test = torch.randn(1, 10, 13)

with torch.no_grad():
    pred = model(test)
    _, predicted = torch.max(pred, 1)

words = ["hello", "yes", "no"]

print("Predicted word:", words[predicted.item()])

Training Done
Predicted word: hello


In [5]:
# mini Voice Typing GUI app using Python
# Install required libraries:
# pip install SpeechRecognition pyaudio

import tkinter as tk
import speech_recognition as sr
import threading

# Function for voice typing
def voice_to_text():
    def process():
        r = sr.Recognizer()

        try:
            with sr.Microphone() as source:
                status_label.config(text="🎤 Listening...")
                root.update()

                r.adjust_for_ambient_noise(source)
                audio = r.listen(source)

                status_label.config(text="🔄 Converting...")
                root.update()

                text = r.recognize_google(audio)

                text_box.delete(1.0, tk.END)
                text_box.insert(tk.END, text)

                status_label.config(text="✅ Done")

        except sr.UnknownValueError:
            status_label.config(text="❌ Could not understand audio")

        except sr.RequestError:
            status_label.config(text="❌ Internet error")

        except Exception as e:
            status_label.config(text="❌ Error: " + str(e))

    # Run in separate thread to avoid UI freezing
    threading.Thread(target=process).start()


# Create GUI window
root = tk.Tk()
root.title("Voice Typing App")
root.geometry("400x300")
root.resizable(False, False)

# Button
btn = tk.Button(root, text="🎙️ Speak", command=voice_to_text, font=("Arial", 14))
btn.pack(pady=20)

# Text box
text_box = tk.Text(root, height=5, width=40, font=("Arial", 12))
text_box.pack()

# Status label
status_label = tk.Label(root, text="Click and speak", fg="blue", font=("Arial", 10))
status_label.pack(pady=10)

# Run app
root.mainloop()

In [4]:
# ext → Speech (TTS) using pyttsx3.
# Install required libraries:
# pip install SpeechRecognition pyttsx3 pyaudio

import tkinter as tk
import speech_recognition as sr
import pyttsx3
import threading

# Initialize text-to-speech engine
engine = pyttsx3.init()
engine.setProperty('rate', 160)

voices = engine.getProperty('voices')
engine.setProperty('voice', voices[1].id)  # try 0 or 1

# Function: Voice → Text → Speech
def voice_assistant():
    def process():
        r = sr.Recognizer()

        try:
            with sr.Microphone() as source:
                status_label.config(text="🎤 Listening...")
                root.update()

                r.adjust_for_ambient_noise(source)
                audio = r.listen(source)

                status_label.config(text="🔄 Converting...")
                root.update()

                text = r.recognize_google(audio)

                # Show text
                text_box.delete(1.0, tk.END)
                text_box.insert(tk.END, text)

                status_label.config(text="✅ Speaking...")

                # Speak the text
                engine.say(text)
                engine.runAndWait()

                status_label.config(text="✅ Done")

        except sr.UnknownValueError:
            status_label.config(text="❌ Could not understand")

        except sr.RequestError:
            status_label.config(text="❌ Internet error")

        except Exception as e:
            status_label.config(text="❌ Error: " + str(e))

    # Run in thread (avoid freeze)
    threading.Thread(target=process).start()


# ---------------- GUI ----------------
root = tk.Tk()
root.title("AI Voice Assistant")
root.geometry("400x350")
root.resizable(False, False)

# Button
btn = tk.Button(root, text="🎙️ Speak", command=voice_assistant, font=("Arial", 14))
btn.pack(pady=20)

# Text box
text_box = tk.Text(root, height=6, width=40, font=("Arial", 12))
text_box.pack()

# Status label
status_label = tk.Label(root, text="Click and speak", fg="blue")
status_label.pack(pady=10)

# Run app
root.mainloop()
